# Treinar a U-Net no Colab (com GPU)

Este notebook baixa os dados do LUNA16, treina e avalia a U-Net 2D de
segmentação pulmonar usando a GPU gratuita do Colab, em vez do CPU local
(que limitava o treino a 40 pacientes e 6 épocas). Com GPU, o próprio
código já aumenta o escopo sozinho (todos os pacientes de treino, 25
épocas, rede maior) — ver `src/luna16/unet.py`.

**Antes de rodar:**
1. Ative a GPU: menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
2. Tenha em mãos um token de API do Kaggle: [kaggle.com/settings](https://www.kaggle.com/settings) → API → *Create New Token* (baixa um arquivo `kaggle.json`).
3. Rode as células em ordem, de cima para baixo.

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Sem GPU -- confira Ambiente de execução > Alterar tipo de ambiente de execução > GPU")

## 1. Trazer o código

**Opção A (recomendada) — se o repositório já foi enviado ao GitHub:** troque
a URL abaixo pela do repositório do grupo e rode a célula.

In [ ]:
REPO_URL = "https://github.com/SEU-USUARIO/PI3-GRUPO-2.git"  # <-- trocar aqui

!git clone -b sprints-3-8 https://github.com/rcoliveirasb/PI3-GRUPO-2.git projeto
%cd projeto

**Opção B — se ainda não subiu pro GitHub:** localmente, compacte só o
código (sem as pastas `data/` e `.venv/`, que são enormes) e suba o `.zip`
pelo ícone de pasta no menu à esquerda do Colab. Depois rode:

```python
!unzip -q SEU_PROJETO.zip -d projeto
%cd projeto
```

(pule a célula da Opção A se for usar esta).

In [ ]:
!pip install -q SimpleITK pydicom scikit-image kaggle

## 2. Configurar acesso ao Kaggle

O Kaggle agora dá um **token de API** direto (formato `KGAT_...`), em vez do
arquivo `kaggle.json` de antes -- gerado em kaggle.com/settings → API →
*Create New Token* (aparece só uma vez, copie na hora).

**Nunca cole o token direto numa célula do notebook** -- este arquivo pode ir
pro GitHub, e o token ficaria exposto pra qualquer um ver. Em vez disso, use
o cofre de segredos do Colab:

1. Clique no ícone de **chave (🔑)** na barra lateral esquerda do Colab.
2. Clique em **"Adicionar novo segredo"**, nome `KAGGLE_TOKEN`, cole o valor do token.
3. Ative o toggle **"Acesso ao notebook"** para esse segredo.
4. Rode a célula abaixo.

In [ ]:
from google.colab import userdata
import os

token = userdata.get("KAGGLE_TOKEN")  # lido do cofre de segredos, nunca fica salvo no notebook
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(token)
os.chmod("/root/.kaggle/access_token", 0o600)
print("Token configurado.")

## 3. Baixar os dados (subset0 + subset1 -- os mesmos 177 pacientes usados localmente)

Cada subset tem ~12GB e leva alguns minutos. Se já tiver rodado antes nesta
sessão do Colab, pode pular (o script não baixa de novo o que já existe).

In [ ]:
!python scripts/download_luna16_subset.py 0
!python scripts/download_luna16_subset.py 1

## 4. Gerar o split 70/15/15 por paciente

In [ ]:
!python scripts/regenerate_split_70_15_15.py

## 5. Treinar a U-Net

Com GPU, o script detecta sozinho e usa mais pacientes/épocas/rede maior
(ver `src/luna16/unet.py` e `scripts/train_unet_baseline.py`). Deve levar
poucos minutos numa T4 do Colab.

In [ ]:
!python scripts/train_unet_baseline.py

## 6. Avaliar a U-Net no conjunto de teste (Dice/IoU)

Roda o mesmo pipeline de avaliação usado para o baseline e o region
growing, garantindo uma comparação justa entre os três métodos.

In [ ]:
!python scripts/run_unet_evaluation.py

## 7. Ver o resultado resumido

In [ ]:
import pandas as pd

df = pd.read_csv("data/luna16/sprint_unet_test.csv")
print(f"Pacientes avaliados: {len(df)}")
print(f"Dice médio: {df['dice'].mean():.4f}  (meta do feedback: >= 0.75)")
print(f"IoU médio:  {df['iou'].mean():.4f}")

## 8. Levar o resultado de volta

Baixe `data/luna16/sprint_unet_test.csv` e `data/luna16/unet_baseline.pt`
pelo ícone de pasta à esquerda (clique com o botão direito no arquivo →
*Download*) e coloque na mesma pasta no projeto local, pra usar nos
gráficos/tabela final e no relatório.